## 1. What a test does

A test calls code with a controlled input and checks an observable result. It should answer one specific question.

```text
Arrange input -> Act by calling code -> Assert the result
```

`pytest` discovers functions whose names begin with `test_` inside files normally named `test_*.py`.

### Completed example using unrelated code

```python
def shipping_cost(order_total: float) -> float:
    if order_total >= 50:
        return 0.0
    return 5.0


def test_shipping_is_free_at_boundary() -> None:
    # Arrange
    order_total = 50.0

    # Act
    result = shipping_cost(order_total)

    # Assert
    assert result == 0.0
```

`assert condition` returns nothing when the condition is true. When false, it raises `AssertionError`. Pytest catches that error and reports the test as failed.

### Test syntax anatomy

```python
def test_behavior_being_checked() -> None:
    actual = function_under_test(input_value)
    assert actual == expected_value
```

| Part | Meaning |
|---|---|
| `test_...` | Makes the function discoverable by pytest |
| `-> None` | The test does not return a useful result |
| `actual` | The result produced by your code |
| `expected_value` | The result required by the contract |
| `assert` | Fail this test if the condition is false |

Tests do not need to call `print()`. A printed result still requires a human to inspect it. An assertion makes the check automatic.

## 2. Add pytest to the project

Pytest is not currently installed in your MLS environment. Declare it as a development dependency instead of making it a runtime requirement.

Add this to the project-root `pyproject.toml`:

```toml
[project.optional-dependencies]
dev = ["pytest>=8,<9"]
```

Meaning:

- `optional-dependencies` groups tools not required by normal package users.
- `dev` is the group name chosen for development tools.
- `>=8,<9` accepts compatible pytest 8 releases while avoiding an unreviewed major upgrade.

From `Code/python_engineering`, run:

```powershell
python -m pip install -e ".[dev]"
python -m pytest --version
```

PowerShell quotes `.[dev]` so it reaches pip as one argument. The dot still means the current project.

### CHECKPOINT 1

Do these exact actions:

1. Confirm the MLS environment is active with `python -c "import sys; print(sys.executable)"`.
2. Add the development dependency.
3. Install the project with the `dev` extra.
4. Run `python -m pytest --version`.

Stop if the executable is not inside the MLS environment or pytest is unavailable.

## 3. Create the first real test file

From `Code/python_engineering`, create the folder and file yourself:

```powershell
New-Item -ItemType Directory -Path .\tests -Force
New-Item -ItemType File -Path .\tests\test_validation.py -Force
```

Write a test for the valid case of `validate_binary_label`.

### Syntax hint

```python
from ml_utils import name_to_test


def test_descriptive_behavior() -> None:
    result = name_to_test(valid_input)
    assert result is None
```

Use `is None`, not `== None`. The validator's successful contract is to finish without raising and return `None`.

Run only that file:

```powershell
python -m pytest tests/test_validation.py -v
```

Expected shape of output:

```text
collected 1 item
tests/test_validation.py::test_... PASSED
```

- `collected 1 item` means discovery found one test.
- The long name identifies the file and test function.
- `PASSED` means the function completed without an uncaught failure.

If pytest reports `collected 0 items`, check the test filename and function name first.

## 4. Testing expected exceptions with `pytest.raises`

An invalid input should raise `ValueError`. That error is expected behavior, so the test passes only when it occurs.

### Completed unrelated example

```python
import pytest


def set_volume(level: int) -> None:
    if level not in range(0, 11):
        raise ValueError("level must be from 0 to 10")


def test_volume_rejects_value_above_ten() -> None:
    with pytest.raises(ValueError, match="level must be"):
        set_volume(11)
```

Syntax flow:

```text
with pytest.raises(expected_error):
    call_that_should_raise()
```

`pytest.raises(...)` returns a context manager. It watches code inside the indented block. The optional `match` checks the error message using a regular-expression pattern.

### Your TODO

Add one test to `test_validation.py` proving that label `2` raises `ValueError`.

Syntax hint:

```python
import pytest

with pytest.raises(EXCEPTION_TYPE, match="stable part of message"):
    function_call(...)
```

Do not catch the error using your own `try/except` in this test. Let `pytest.raises` perform the assertion.

### Common failure output

```text
Failed: DID NOT RAISE <class 'ValueError'>
```

Meaning: the call completed normally even though the test required `ValueError`.

```text
AssertionError: Regex pattern did not match
```

Meaning: the expected exception type occurred, but its message did not match your `match` pattern. Prefer a short, stable part of the message rather than the entire sentence.

### CHECKPOINT 2

Run:

```powershell
python -m pytest tests/test_validation.py -v
```

Required result: two tests pass. One covers a valid label and one covers an invalid label.

## 5. Normal, boundary, and invalid cases

A useful test suite samples different parts of a function's contract.

| Case | Question | `TrainingConfig` example |
|---|---|---|
| Normal | Does ordinary valid input work? | learning rate `0.01` |
| Boundary | Does the exact allowed edge work? | threshold `0.0` or `1.0` |
| Invalid | Is forbidden input rejected? | learning rate `0.0` |

Boundary tests matter because mistakes often use `<` when they need `<=`, or the reverse.

Create `tests/test_config.py`. Write these tests yourself:

1. A normal valid configuration stores all three values.
2. Threshold `0.0` is accepted.
3. Threshold `1.0` is accepted.
4. Learning rate `0.0` raises `ValueError`.
5. Epochs `0` raises `ValueError`.
6. Threshold above `1.0` raises `ValueError`.

Use direct attribute assertions for valid instances and `pytest.raises` for invalid construction.

### Attribute-test syntax

```python
def test_object_stores_value() -> None:
    instance = ClassName(required_value, another_value)
    assert instance.attribute_name == expected_value
```

This checks observable behavior. Do not test dataclass implementation details such as generated internal methods.

## 6. Test the existing accuracy metric

### The target

Create `tests/test_metrics.py` and prove that `calculate_accuracy()` follows its contract.

The function receives two lists:

```text
y_true = correct labels
y_pred = model predictions
```

It should validate both lists and return the fraction of matching positions as a `float`.

```text
[1, 0, 1] compared with [1, 0, 0]
     match              match

2 matches / 3 positions = 0.6666...
```

### Why this matters

Accuracy may look simple, but wrong lengths, empty lists, invalid labels, and floating-point comparisons can silently produce bad evaluation results. We will test one behavior at a time so each failure has one clear meaning.

### Step 6.1: perfect predictions

**Target:** Prove that identical labels produce accuracy `1.0`.

**File:** `tests/test_metrics.py`

**Different completed example:**

```python
def percentage_complete(done: int, total: int) -> float:
    return done / total


def test_percentage_is_complete_when_all_work_is_done() -> None:
    result = percentage_complete(4, 4)
    assert result == 1.0
```

The test arranges `4` completed tasks out of `4`, calls the function, then checks its returned float.

**Your single task:** Create `test_metrics.py`. Import `calculate_accuracy`, call it with two identical binary-label lists, and assert that it returns `1.0`.

**Syntax hint:**

```python
from ml_utils import name_to_import


def test_descriptive_behavior() -> None:
    result = function_call(first_list, second_list)
    assert result == expected_float
```

**Run:**

```powershell
python -m pytest tests/test_metrics.py -v
```

**Expected output:** Pytest collects one test and marks it `PASSED`.

**Learning outcome:** You can test an ordinary return value using `assert`.

### Step 6.2: partial accuracy and `pytest.approx`

**Target:** Prove that two correct predictions out of three produce approximately `2 / 3`.

**Why:** Values such as one third cannot be represented exactly as a finite binary floating-point number.

**Different completed example:**

```python
import pytest


def test_one_third_is_approximate() -> None:
    result = 1 / 3
    assert result == pytest.approx(0.3333333333333333)
```

`pytest.approx(expected)` returns a comparison helper. It accepts a sufficiently close floating-point value. It does not modify `result`.

**Your single task:** Add one test using labels where exactly two of three positions match. Compare the result with `pytest.approx(2 / 3)`.

**Syntax hint:**

```python
import pytest

assert actual_float == pytest.approx(expected_float)
```

**Run:**

```powershell
python -m pytest tests/test_metrics.py -v
```

**Expected output:** Two tests are collected and both pass.

**Learning outcome:** You can compare calculated floats without depending on exact representation.

### Step 6.3: reject empty lists

**Target:** Prove that empty inputs raise `ValueError` instead of causing division by zero or returning a misleading result.

**File:** `tests/test_metrics.py`

**Different completed example:**

```python
import pytest


def first_item(items: list[str]) -> str:
    if not items:
        raise ValueError("items cannot be empty")
    return items[0]


def test_first_item_rejects_empty_list() -> None:
    with pytest.raises(ValueError, match="cannot be empty"):
        first_item([])
```

The context manager returned by `pytest.raises()` watches the indented call. The test fails if the expected exception is not raised.

**Your single task:** Add one test that calls `calculate_accuracy([], [])` inside `pytest.raises(ValueError)`.

**Expected output:** Three tests pass.

**Learning outcome:** You can verify that invalid input fails in the intended way.

### Step 6.4: reject unequal lengths

**Target:** Prove that every true label must have exactly one predicted label.

**Why:** Comparing lists of different lengths would make the metric incomplete or misleading.

**Different completed example:**

```python
def pair_names(names: list[str], scores: list[int]) -> None:
    if len(names) != len(scores):
        raise ValueError("lengths must match")
```

`len(first) != len(second)` produces a Boolean. `True` means the precondition failed.

**Your single task:** Add one expected-exception test using two non-empty label lists with different lengths.

**Syntax hint:** Reuse the `pytest.raises` structure from Step 6.3. Do not add new production code because `calculate_accuracy()` already performs this validation.

**Expected output:** Four tests pass.

**Learning outcome:** You understand why aligned ML labels require equal lengths.

### Step 6.5: reject an invalid true label

**Target:** Prove that `y_true` may contain only `0` and `1`.

**Different completed example:**

```python
def validate_direction(direction: str) -> None:
    if direction not in ("left", "right"):
        raise ValueError("invalid direction")
```

Membership validation asks whether a value belongs to the allowed set.

**Your single task:** Add one test with an invalid value such as `2` in `y_true` and a valid `y_pred`. Require `ValueError`.

**Expected output:** Five tests pass.

**Learning outcome:** Your test proves that ground-truth data is validated before a metric is trusted.

### Step 6.6: reject an invalid predicted label

**Target:** Prove that `y_pred` also accepts only `0` and `1`.

**Why:** Testing only `y_true` would leave half of the function's inputs unprotected.

**Different completed example:**

```python
def test_second_argument_is_checked() -> None:
    with pytest.raises(ValueError):
        compare_values(valid_first_value, invalid_second_value)
```

**Your single task:** Add one test with valid `y_true` and an invalid label in `y_pred`.

**Run:**

```powershell
python -m pytest tests/test_metrics.py -v
```

**Expected output:** Six tests are collected from this file and all pass.

**Learning outcome:** You can check the same contract independently for two parameters.

### Checkpoint: read the result

Run the complete suite:

```powershell
python -m pytest -q
```

Expected shape:

```text
..........                                                       [100%]
N passed in ...s
```

Your value of `N` depends on how many earlier tests you wrote. The important facts are `[100%]` and no failures.

If something fails, read in this order:

1. Final exception or assertion message.
2. Failing test name.
3. Highlighted source line.
4. Actual and expected values.

Do not edit production code automatically. First decide whether the implementation is wrong or the test expects the wrong behavior.

## 7. What a regression test protects

### Target

Understand how an ordinary behavior test prevents an old bug from returning.

Earlier, the loop in `calculate_accuracy()` used `rang`, a name that did not exist:

```python
for i in range(rang):
```

Python could import the module because the bad name was inside the function body. The failure appeared only when a call reached that loop:

```text
test calls calculate_accuracy
    -> validation passes
        -> loop tries to read rang
            -> NameError
                -> test fails
```

After changing it to the correct existing length variable, the normal accuracy test passes.

```text
reproduce bug -> write or identify failing test -> fix code -> keep test
```

### Why this matters

The permanent value is not the one-time fix. It is the test that automatically detects the same broken behavior later.

### Naming rule

Name the required behavior, not the historical typo:

```text
Good: test_accuracy_counts_matching_labels
Weak: test_rang_typo_is_fixed
```

### Learning outcome

You can explain why a regression test describes expected behavior and remains useful after the original bug is forgotten.

## 8. Capstone overview: what are we building?

### Final target

Build one small binary-classification evaluator. It receives:

```text
1. y_true: the correct 0/1 labels
2. probabilities: model confidence values from 0.0 to 1.0
3. config: settings containing the classification threshold
```

It processes them like this:

```text
probabilities
    -> validate every probability
    -> convert each probability to prediction 0 or 1
    -> compare predictions with y_true
    -> calculate accuracy and confusion counts
    -> return one dictionary
```

Example mental model, not your TODO answer:

```text
threshold = 0.50
probabilities = [0.80, 0.20]
predictions   = [1,    0]
```

The final result has these keys:

```text
accuracy
true_positive
true_negative
false_positive
false_negative
```

### Why this matters

This is a small version of real model evaluation code. It connects validation, configuration, modules, return values, type hints, imports, and tests.

## 9. Layer A: validate one probability

### Function contract

```python
validate_probability(probability: float) -> None
```

| Part | Meaning |
|---|---|
| `probability` | One numeric confidence value |
| Valid range | `0.0` through `1.0`, including both boundaries |
| Valid return | `None` |
| Invalid behavior | Raise `ValueError` |

### Different completed example

```python
def validate_percentage(value: float) -> None:
    if value < 0.0 or value > 100.0:
        raise ValueError("percentage must be from 0 to 100")
```

The function has no useful value to return. Its job is to stop invalid data.

### Step 9.1: implement the validator

**Target:** Add `validate_probability()`.

**File:** `src/ml_utils/validation.py`

**Your single task:** Write the function contract above. Reject values below `0.0` or above `1.0` with a clear `ValueError` message.

**Syntax hint:**

```python
def function_name(value: float) -> None:
    if value < LOWER_BOUND or value > UPPER_BOUND:
        raise ValueError("clear message")
```

**Quick manual check:**

```powershell
python -c "from ml_utils.validation import validate_probability; print(validate_probability(0.5))"
```

**Expected output:** `None`

**Learning outcome:** You can express an inclusive numeric precondition.

### Step 9.2: test one normal probability

**Target:** Prove that `0.5` is accepted.

**File:** `tests/test_validation.py`

**Different completed example:**

```python
def test_normal_percentage_is_accepted() -> None:
    result = validate_percentage(50.0)
    assert result is None
```

**Your single task:** Import `validate_probability`, call it with `0.5`, and assert that the returned value is `None`.

**Expected output:** The new test passes.

**Learning outcome:** You tested the normal path separately from edge cases.

### Step 9.3: test both valid boundaries

**Target:** Prove that `0.0` and `1.0` are valid because the contract says inclusive.

**File:** `tests/test_validation.py`

**Different completed example:**

```python
def test_percentage_boundaries_are_accepted() -> None:
    assert validate_percentage(0.0) is None
    assert validate_percentage(100.0) is None
```

**Your single task:** Write one boundary test containing two assertions for `0.0` and `1.0`.

**Expected output:** The boundary test passes. If it fails, check `<` versus `<=` in production code.

**Learning outcome:** You can translate the word “inclusive” into boundary tests.

### Step 9.4: test invalid probabilities

**Target:** Prove that values on both sides of the allowed range are rejected.

**Different completed example:**

```python
def test_percentages_outside_range_are_rejected() -> None:
    with pytest.raises(ValueError):
        validate_percentage(-0.1)
    with pytest.raises(ValueError):
        validate_percentage(100.1)
```

**Your single task:** Add one test for a value below `0.0` and one for a value above `1.0`.

**Run:**

```powershell
python -m pytest tests/test_validation.py -v
```

**Expected output:** All old and new validation tests pass.

**Learning outcome:** You protected both invalid directions rather than testing only one.

## 10. Layer B: calculate confusion counts

### Function contract

```python
confusion_counts(
    y_true: list[int],
    y_pred: list[int],
) -> dict[str, int]
```

It returns exactly four integer counts:

| Actual | Predicted | Count |
|---:|---:|---|
| 1 | 1 | `true_positive` |
| 0 | 0 | `true_negative` |
| 0 | 1 | `false_positive` |
| 1 | 0 | `false_negative` |

“True” means the prediction was correct. “Positive” means the predicted class was `1`.

### Completed example: pair two lists safely

```python
names = ["A", "B"]
scores = [80, 90]

for name, score in zip(names, scores):
    print(name, score)
```

Output:

```text
A 80
B 90
```

`zip(names, scores)` returns an iterator producing pairs. It silently stops at the shorter list, so validate equal lengths before using it.

Dictionary increment syntax:

```python
counts["some_key"] += 1
```

This reads the current integer, adds one, and stores the new integer under the same key.

### Step 10.1: initialize and return all counts

**Target:** Create the function and always return all four keys, initially with zero values.

**File:** `src/ml_utils/metrics.py`

**Your single task:** Add the typed function signature and initialize a `dict[str, int]` containing the four required keys. Return that dictionary after the future loop location.

**Syntax hint:**

```python
counts: dict[str, int] = {
    "first_key": 0,
    # remaining keys
}
```

Do not implement category comparisons in this step.

**Learning outcome:** You can construct a stable result shape where missing categories still appear as zero.

### Step 10.2: add validation before pairing

**Target:** Reject empty lists, unequal lengths, and invalid labels before counting.

**Why:** Once counting begins, every pair must be trustworthy.

**Your single task:** Reuse the same validation ideas already used by `calculate_accuracy()`. Do not copy the entire accuracy calculation.

**Syntax hints:**

```python
if EMPTY_OR_LENGTH_CONDITION:
    raise ValueError("clear message")

for label in some_list:
    validate_binary_label(label)
```

**Expected behavior:** Valid lists reach the counting loop. Invalid lists raise before any result is returned.

**Learning outcome:** You place precondition checks before the main operation.

### Step 10.3: count true positives

**Target:** Increment only `true_positive` when actual and predicted labels are both `1`.

**Different completed example:**

```python
for expected, actual in zip(expected_colors, actual_colors):
    if expected == "green" and actual == "green":
        counts["both_green"] += 1
```

**Your single task:** Add the loop and the true-positive condition. Do not add the other three branches yet.

**Expected manual result:** For `y_true=[1]` and `y_pred=[1]`, `true_positive` becomes `1` while the other keys remain `0`.

**Learning outcome:** You can map one pair of labels to one confusion category.

### Step 10.4: add the other three categories

Add one branch, save, and reason through one pair before adding the next.

**Task A:** Actual `0`, predicted `0` increments `true_negative`.

**Task B:** Actual `0`, predicted `1` increments `false_positive`.

**Task C:** Actual `1`, predicted `0` increments `false_negative`.

Syntax shape:

```python
if FIRST_CONDITION:
    ...
elif SECOND_CONDITION:
    ...
elif THIRD_CONDITION:
    ...
else:
    ...
```

Exactly one branch should execute for every valid binary pair.

**Learning outcome:** You can translate a four-row decision table into mutually exclusive branches.

### Step 10.5: test one of each category

**Target:** One small dataset should contain one true positive, one true negative, one false positive, and one false negative.

**File:** `tests/test_metrics.py`

**Different completed example:**

```python
def test_color_summary_contains_each_category() -> None:
    result = summarize_colors([...], [...])
    assert result["both_green"] == 1
    assert result["both_red"] == 1
```

**Your single task:** Design four label pairs on paper, call `confusion_counts`, and assert each returned key equals `1`.

Do not use `confusion_counts()` itself to calculate your expected values.

**Expected output:** The new confusion-count test passes.

**Learning outcome:** You can independently calculate expected output for a test.

### Step 10.6: test confusion-count failures

Add these one at a time and run the file after each:

1. Empty lists raise `ValueError`.
2. Unequal lengths raise `ValueError`.
3. An invalid label raises `ValueError`.

Each test reuses `pytest.raises`; no new test syntax is introduced.

```powershell
python -m pytest tests/test_metrics.py -v
```

**Expected output:** All accuracy and confusion-count tests pass.

**Learning outcome:** A new metric follows the same input contract as the existing metric.

## 11. Layer C: assemble the evaluator

### Function contract

Create `src/ml_utils/evaluator.py`:

```python
evaluate_binary_classifier(
    y_true: list[int],
    probabilities: list[float],
    config: TrainingConfig,
) -> dict[str, float | int]
```

### Parameter meanings

- `y_true`: known correct binary labels.
- `probabilities`: model confidence for class `1`.
- `config`: provides the threshold and represents the evaluation settings.

### Return value

One dictionary containing accuracy as a float and four confusion counts as integers.

The evaluator coordinates existing helpers. It should not reimplement their calculations.

### Step 11.1: create the module and imports

**Target:** Create the function structure without its processing body.

**File:** `src/ml_utils/evaluator.py`

**Your single task:** Import `TrainingConfig`, `calculate_accuracy`, `confusion_counts`, and `validate_probability` using absolute imports. Add the evaluator signature and docstring.

**Syntax pattern:**

```python
from package_name.module_name import public_name


def function_name(parameter: type) -> return_type:
    '''Explain the input-to-output purpose.'''
    # body comes in later steps
```

**Learning outcome:** You can declare how data will flow across package modules before implementing the flow.

### Step 11.2: validate lengths and probabilities

**Target:** Reject inputs that cannot form one probability for every true label.

**Your tasks, one at a time:**

1. Reject empty inputs.
2. Reject unequal lengths.
3. Loop through `probabilities` and call `validate_probability()` for each value.

Run a small manual import after each saved change. Do not build predictions until these checks behave correctly.

**Learning outcome:** The coordinator validates its own probability-list contract before delegating metric work.

### Step 11.3: convert probabilities into labels

**Target:** Produce `predictions: list[int]` using `config.threshold`.

Rule:

```text
probability >= threshold -> 1
probability < threshold  -> 0
```

### Different completed example

```python
temperatures = [18, 25, 30]
labels = ["hot" if value >= 25 else "cool" for value in temperatures]
```

The list comprehension iterates over each temperature, evaluates the condition, and appends one selected value to a new list.

**Your single task:** Write the equivalent comprehension for probabilities and `config.threshold`.

**Syntax hint:**

```python
new_list = [VALUE_IF_TRUE if CONDITION else VALUE_IF_FALSE for item in items]
```

**Learning outcome:** You can apply one classification threshold to a sequence of probabilities.

### Step 11.4: call the two metrics

**Target:** Reuse existing functions instead of recalculating their logic.

**Your single task:** Pass `y_true` and the new predictions into `calculate_accuracy()` and `confusion_counts()`. Store both returned values in clearly named variables.

**Syntax pattern:**

```python
first_result = first_function(shared_input, processed_input)
second_result = second_function(shared_input, processed_input)
```

**Learning outcome:** You can compose small tested functions into a larger workflow.

### Step 11.5: build and return the result dictionary

**Target:** Combine accuracy and the four counts into one dictionary.

### Different completed example

```python
summary: dict[str, float | int] = {"average": 8.5}
counts = {"passed": 3, "failed": 1}
summary.update(counts)
return summary
```

`dict.update(other)` adds or replaces entries in the existing dictionary and returns `None`.

**Your single task:** Create the result with the `accuracy` key, update it using the confusion-count dictionary, then return the result.

Do not write:

```python
return result.update(counts)
```

That would return `None`.

**Learning outcome:** You understand the difference between a mutating method's side effect and its return value.

## 12. Layer D: public exports

### Target

Make the three new functions available through the package interface:

```python
from ml_utils import (
    validate_probability,
    confusion_counts,
    evaluate_binary_classifier,
)
```

### File

`src/ml_utils/__init__.py`

### Your task

Add one absolute import for each public function. Do not add calls, calculations, or print statements.

### Check

```powershell
python -c "from ml_utils import validate_probability, confusion_counts, evaluate_binary_classifier; print('imports work')"
```

Expected output:

```text
imports work
```

### Learning outcome

You can intentionally extend a package's public interface without creating import side effects.

## 13. End-to-end evaluator tests

Create `tests/test_evaluator.py`. Each checkpoint below adds one behavior.

### Step 13.1: normal evaluation

**Target:** Prove that the evaluator returns the correct accuracy and all four counts.

**Different completed example:**

```python
def test_summary_combines_average_and_counts() -> None:
    result = build_summary([8, 10])
    assert result["average"] == 9.0
    assert result["count"] == 2
```

**Your single task:** Choose a small `y_true`, probability list, and threshold. Convert probabilities to predictions on paper. Calculate the five expected results yourself, then assert every key.

**Expected output:** The normal evaluator test passes.

**Learning outcome:** You can test a composed workflow using independently calculated expected results.

### Step 13.2: threshold boundary

**Target:** Prove that a probability exactly equal to the threshold becomes label `1`.

**Why:** The contract uses `>=`, not `>`.

**Your single task:** Create the smallest useful test containing a probability equal to the configured threshold. Assert the confusion result that proves prediction `1` was produced.

**Expected output:** The boundary test passes. A mistaken `>` implementation would fail it.

**Learning outcome:** You can turn one comparison symbol in a contract into a focused boundary test.

### Step 13.3: invalid probability

**Target:** Prove that the evaluator propagates `ValueError` from probability validation.

**Your single task:** Pass one probability outside `0.0` to `1.0` and require `ValueError` using `pytest.raises`.

**Expected output:** The test passes because the error is expected.

**Learning outcome:** You understand that a coordinator does not need to catch an exception when the caller should see the invalid input.

### Step 13.4: unequal lengths

**Target:** Prove that every true label requires one probability.

**Your single task:** Use non-empty lists with different lengths and require `ValueError`.

**Expected output:** The test passes only if the evaluator rejects the mismatch before evaluation.

**Learning outcome:** You can test a precondition at the public workflow boundary.

### Step 13.5: deterministic result

**Target:** Prove that identical inputs produce equal output dictionaries.

**Different completed example:**

```python
def test_formatter_is_repeatable() -> None:
    first = format_record("A")
    second = format_record("A")
    assert first == second
```

**Your single task:** Call the evaluator twice with the same inputs and configuration. Assert that the two returned dictionaries are equal.

**Expected output:** The test passes because this evaluator contains no randomness or external state.

**Learning outcome:** You can express deterministic behavior as a repeatable test.

## 14. Final acceptance check

### Target

Prove that the complete package works, not only the most recently edited file.

Run from `Code/python_engineering`:

```powershell
python -m pytest -q
python -c "import ml_utils; print('clean import')"
python -m ml_utils
git status --short
git diff --check
```

### Expected results

- Pytest reaches `[100%]` with all tests passing.
- Import prints only `clean import`.
- `python -m ml_utils` still prints the earlier demonstration.
- `git status --short` shows only files you intended to change.
- `git diff --check` prints nothing when no whitespace errors exist.

### What you should now be able to do

```text
write a function contract
-> validate inputs
-> return a typed value
-> expose it through a package
-> test normal, boundary, and invalid behavior
-> combine tested functions into a small ML workflow
```

Send the complete pytest output and changed files for review before committing.

## Final reflection: three questions only

Answer after the complete suite passes. Use one or two sentences each.

In [ ]:
final_reflection = {
    "How does a pytest test turn expected behavior into an automatic pass or failure?": "",
    "Why did we test normal, boundary, and invalid inputs separately?": "",
    "How does the evaluator reuse validation and metric functions instead of duplicating their logic?": "",
}

## Retrieval task: complete after 2–3 days

Without reopening this notebook, write one normal test, one boundary test, and one expected-exception test for a tiny function. Run them and record only the command and final result.

In [ ]:
retrieval_date = ""
retrieval_command = ""
retrieval_result = ""

## Stop

Do not complete the retrieval task today. Session 5 is complete when the capstone suite passes and you can explain each test without relying on its comments.